In [ ]:
import sys; print(sys.executable)
!jupyter kernelspec list
!jupyter kernelspec install /usr/local/share/jupyter/kernels/python312-ml-cpu --user --name orange-custom-kernel

"""
Duplicated the "/usr/local/share/jupyter/kernels/python312-ml-cpu" kernel by
jupyter kernelspec install /usr/local/share/jupyter/kernels/python312-ml-cpu --user --name orange-custom-kernel
Added  ",
"env": {
  "PYTHONPATH": "/stor/home/he4249/orange/my_custom_packages"
}"
And chose the kernel.
This is so n_jobs = -1 works and every child created inherits the PYTHONPATH to the libraries.
"""

In [ ]:
import sys
import os
#!pip install polars --target ./my_custom_packages
#sys.path.append("./my_custom_packages") # scikit-learn, causalml, optuna, kmodes, kneed, joblib
custom_path = "/stor/home/he4249/orange/my_custom_packages"
os.environ["PYTHONPATH"] = custom_path
sys.path.insert(0, custom_path)

import polars as pl
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns

import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler, OneHotEncoder

import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
from sklearn.metrics import classification_report

from sklearn.neighbors import NearestNeighbors

from causalml.match import NearestNeighborMatch
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import optuna

# numpy 2.4
from causalml.inference.meta import BaseTRegressor
from causalml.metrics import plot_qini

from kmodes.kprototypes import KPrototypes

from kneed import KneeLocator

import joblib


In [ ]:
df_merged =  pl.read_parquet("df_merged.parquet")
display(df_merged.head())

In [ ]:

df_psm = df_merged.group_by("msisdn").agg(
    pl.col("CA").sum().alias("UTCA"),
    pl.len().alias("n_transactions"),
    pl.col("Nom du forfait").n_unique().alias("nunique_mobile_plan"),
    pl.col("Gamme_groupe").n_unique().alias("nunique_plan_type"),
    pl.col("Group_canal").mode().first().alias("prefered_channel"),
    pl.col("Max_RAT").first().alias("prefered_data_tech"),
    pl.col("Typologie").first().alias("prefered_geographic_classification"),
    pl.col("region_cleaned").first().alias("prefered_region"),
    pl.col("nb_jours").first().alias("connectivity_stability"),
    (pl.col("Nom du forfait").n_unique() > 1).cast(pl.Int8).alias("is_multi_plan_user"),
    pl.col("Gamme_groupe").is_in(["Akama/Be Connect"]).any().cast(pl.Int8).alias("buys_data"),
    pl.col("Gamme_groupe").is_in(["Be"]).any().cast(pl.Int8).alias("buys_voice"),
    pl.col("Gamme_groupe").is_in(["Be Sms"]).any().cast(pl.Int8).alias("buys_sms"),
    pl.col("Group_canal").is_in(["APP OM", "MAXIT"]).any().cast(pl.Int8).alias("app_user"),
    pl
        .when(pl.col("Gamme_groupe").is_in(["Be"]).any())
        .then(pl.col("Gamme_groupe").is_in(["Akama/Be Connect"]).sum() / pl.col("Gamme_groupe").is_in(["Be"]).sum())
        .otherwise(pl.col("Gamme_groupe").is_in(["Akama/Be Connect"]).sum() + 10)
        .alias("data/voice__ratio"),
    (pl.col("Nom du forfait").len() / pl.col("nb_jours").first()).alias("plans_bought/connectivity__ratio"),
    (pl.col("CA").sum() / pl.col("Nom du forfait").len()).alias("avg_transaction_ca")
).sort("UTCA", descending = True).to_pandas().set_index("msisdn")

#df_psm["UTCA"].describe(percentiles = [0.8])
#round((df_psm.filter(df_psm["UTCA"] >= 8600)["UTCA"].round(decimals= 2).sum() / df_psm["UTCA"].round(decimals= 2).sum()), 2)
# 0.66%

df_psm["VIP"] = (df_psm["UTCA"] >= 8600).astype(int)

df_psm["n_transactions"] = np.log1p(df_psm["n_transactions"])


display(df_psm.head())
display(df_psm.describe())



In [ ]:
# Checking distribution to apply correct scaling.
'''
display(df_psm.describe())
stats.probplot(df_psm["n_transactions"], dist = "norm", plot = plt)
plt.title("n_transactions")
plt.show()

sns.histplot(df_psm["n_transactions"], kde = True)
plt.title("n_transactions")
plt.show()


stats.probplot(df_psm["nunique_mobile_plan"], dist = "norm", plot = plt)
plt.title("nunique_mobile_plan")
plt.show()

sns.histplot(df_psm["nunique_mobile_plan"], kde = True)
plt.title("nunique_mobile_plan")
plt.show()

stats.probplot(df_psm["nunique_plan_type"], dist = "norm", plot = plt)
plt.title("nunique_plan_type")
plt.show()

sns.histplot(df_psm["nunique_plan_type"], kde = True)
plt.title("nunique_plan_type")
plt.show()

stats.probplot(df_psm["connectivity_stability"], dist = "norm", plot = plt)
plt.title("connectivity_stability")
plt.show()

sns.histplot(df_psm["connectivity_stability"], kde = True)
plt.title("connectivity_stability")
plt.show()'''



In [ ]:

x = df_psm.drop(
    ["UTCA", "VIP", "n_transactions", "nunique_plan_type", "nunique_mobile_plan", "app_user", "data/voice__ratio", "plans_bought/connectivity__ratio", "avg_transaction_ca"]
    , axis = 1)
y = df_psm["VIP"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 4, stratify = y)


scaler_std = StandardScaler()
cols_to_scale = ["connectivity_stability"] # , "n_transactions", "nunique_plan_type", "nunique_mobile_plan"

x_train[cols_to_scale] = scaler_std.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler_std.transform(x_test[cols_to_scale])

encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore").set_output(transform = "pandas")
cols_to_encod = ["prefered_channel", "prefered_data_tech", "prefered_geographic_classification", "prefered_region"]

enc_train = encoder.fit_transform(x_train[cols_to_encod])
x_train = x_train.drop(cols_to_encod, axis = 1).join(enc_train)

enc_test = encoder.transform(x_test[cols_to_encod])
x_test = x_test.drop(cols_to_encod, axis = 1).join(enc_test)

x_train.head()


In [ ]:

lr = LogisticRegression(penalty = None)
lr.fit(x_train, y_train)

y_predic_lr = lr.predict(x_test)
display(classification_report(y_test, y_predic_lr))
print(classification_report(y_test, y_predic_lr))

# 17.0s

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators = 150,
    max_depth = 6,
    learning_rate = 0.1,
    n_jobs = -1,
    tree_method = "hist",
    random_state = 4
)

xgb_model.fit(x_train, y_train)

# Gain measures the improvement in accuracy a feature brings to the branches it splits

xgb.plot_importance(xgb_model, importance_type = 'gain')

plt.title("Feature xgboost importance score in determining VIP category (top 20% spending user)")
plt.show()

y_predic_xgb = xgb_model.predict(x_test)
display(classification_report(y_test, y_predic_xgb))
print(classification_report(y_test, y_predic_xgb))

# 3.6s

In [ ]:
# optuna don't forget scale_pos_weight

x = df_psm.drop([
    "UTCA", "VIP", "n_transactions", "nunique_plan_type", "nunique_mobile_plan",
    "app_user", "is_multi_plan_user", "buys_data", "buys_voice", "buys_sms", "prefered_data_tech",
    "data/voice__ratio", "plans_bought/connectivity__ratio", "avg_transaction_ca"
    ], axis = 1)
y = df_psm["VIP"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 4, stratify = y)


# Setting total server cores = (Optuna n_jobs) * (XGBoost n_jobs)
OPTUNA_TRIALS_AT_ONCE = 14
XGBOOST_CORES_PER_TRIAL = 8

cv_strategy = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 4)

cols_to_scale = ["connectivity_stability"]
cols_to_encod = ["prefered_channel", "prefered_geographic_classification", "prefered_region"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical_to_scale", StandardScaler(), cols_to_scale),
        ("caterogical_to_encode", OneHotEncoder(sparse_output = False, handle_unknown = "ignore"), cols_to_encod)
    ],
    remainder = "passthrough" # other features
)

# class imbalance for scale_pos_weight
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

def objective(trial):

    param = {
        "max_depth": trial.suggest_int("max_depth", 1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.5, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }

    model = xgb.XGBClassifier(
        **param,
        objective = "binary:logistic",
        scale_pos_weight = ratio,
        tree_method = "hist",
        random_state = 4,
        eval_metric = "logloss",
        n_jobs = XGBOOST_CORES_PER_TRIAL
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_val_score(
        pipeline,
        x_train,
        y_train,
        cv = cv_strategy,
        scoring = "neg_log_loss", # penalizing false predictions
        n_jobs = 1
    )

    return scores.mean()

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 150, n_jobs = OPTUNA_TRIALS_AT_ONCE)

print(f"Best log loss score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

In [ ]:
# Non VIPs with the highest scores are users with the footprint of a high spender but who aren't spending yet which is untapped potential.

x = df_psm.drop([
    "UTCA", "VIP", "n_transactions", "nunique_plan_type", "nunique_mobile_plan",
    "app_user", "is_multi_plan_user", "buys_data", "buys_voice", "buys_sms", "prefered_data_tech",
    "data/voice__ratio", "plans_bought/connectivity__ratio", "avg_transaction_ca"
    ], axis = 1)
y = df_psm["VIP"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 4, stratify = y)


cols_to_scale = ["connectivity_stability"]
cols_to_encod = ["prefered_channel", "prefered_geographic_classification", "prefered_region"]

scaler_std = StandardScaler()
x_train[cols_to_scale] = scaler_std.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler_std.transform(x_test[cols_to_scale])

encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore").set_output(transform = "pandas")

enc_train = encoder.fit_transform(x_train[cols_to_encod])
x_train = x_train.drop(cols_to_encod, axis = 1).join(enc_train)

enc_test = encoder.transform(x_test[cols_to_encod])
x_test = x_test.drop(cols_to_encod, axis = 1).join(enc_test)


xgb_model = xgb.XGBClassifier(
    max_depth = 12,
    learning_rate = 0.01652328144004374,
    n_estimators = 297,
    subsample = 0.101014017386578,
    colsample_bytree = 0.8755549232762643,
    min_child_weight = 1,
    reg_alpha = 0.16102604685773036,
    reg_lambda = 0.0004784037496382744,
    n_jobs = -1,
    tree_method = "hist",
    eval_metric = "logloss",
    random_state = 4
)

xgb_model.fit(x_train, y_train)


y_predic_xgb = xgb_model.predict(x_test)

display(classification_report(y_test, y_predic_xgb))
print(classification_report(y_test, y_predic_xgb))

# 13.4s

In [ ]:
x_full = df_psm.drop([
    "UTCA", "VIP", "n_transactions", "nunique_plan_type", "nunique_mobile_plan",
    "app_user", "is_multi_plan_user", "buys_data", "buys_voice", "buys_sms", "prefered_data_tech",
    "data/voice__ratio", "plans_bought/connectivity__ratio", "avg_transaction_ca"
    ], axis = 1)
cols_to_scale = ["connectivity_stability"]
cols_to_encod = ["prefered_channel", "prefered_geographic_classification", "prefered_region"]
scaler_std = StandardScaler()
x_full[cols_to_scale] = scaler_std.fit_transform(x_full[cols_to_scale])
encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore").set_output(transform = "pandas")
enc_x = encoder.fit_transform(x_full[cols_to_encod])
x_full = x_full.drop(cols_to_encod, axis = 1).join(enc_x)



proba_vip = xgb_model.predict_proba(x_full)[:, 1]

df_psm_prb = df_psm.copy()
df_psm_prb["proba_vip"] = proba_vip

display(df_psm_prb[df_psm_prb["VIP"] == 0].sort_values("proba_vip", ascending = False)["proba_vip"].head())
display(df_psm_prb["proba_vip"].describe())
display(len(df_psm_prb[df_psm_prb["proba_vip"] >= 0.80]))

# 51167 non vips that act at 80% like a vip.

In [ ]:
%%time

# PROPENSITY SCORE MATCHING: causal inference

# Does the use of app cause users to spend more money? (user to user comparaison)

'''
Underlying profile of someone who uses a Web App in Madagascar:
They must have a smartphone capable of downloading and running the app likely a 4G+ or 5G device.
They likely live in an urban center with strong internet coverage like Antananarivo.
They likely have a bank account or mobile money set up.
Now think about the profile of a strict USSD user:
They might be using a basic feature phone.
They might live in a rural area with intermittent coverage.
We need to block for counfonders.
'''


y = df_psm["app_user"]
x = df_psm.drop(
    ["UTCA", "prefered_channel", "VIP", "n_transactions", "nunique_plan_type", "nunique_mobile_plan", "app_user"]
    , axis = 1)

scaler_std = StandardScaler()
cols_to_scale = ["connectivity_stability", "data/voice__ratio", "plans_bought/connectivity__ratio", "avg_transaction_ca"]
x[cols_to_scale] = scaler_std.fit_transform(x[cols_to_scale])

encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore").set_output(transform = "pandas")
cols_to_encod = ["prefered_data_tech", "prefered_geographic_classification", "prefered_region"]
enc_train = encoder.fit_transform(x[cols_to_encod])
x = x.drop(cols_to_encod, axis = 1).join(enc_train)


lr = LogisticRegression(penalty = None)
lr.fit(x, y)
psm = lr.predict_proba(x)



In [ ]:
%%time

df_psm["propensity_score"] = psm[ : , 1]

df_p = df_psm[df_psm["app_user"] == 1]
df_n = df_psm[df_psm["app_user"] == 0]

nn = NearestNeighbors(n_neighbors = 1, metric = "euclidean").fit(df_n[["propensity_score"]])
dists, indices = nn.kneighbors(df_p[["propensity_score"]])

# 2011 paper by biostatistician Peter C. Austin, "Optimal caliper widths for propensity score matching"
caliper = 0.2 * df_psm["propensity_score"].std()

matches_df = pd.DataFrame({
    "treat_idx": df_p.index,
    "control_idx": df_n.iloc[indices.flatten()].index,
    "distance": dists.flatten()
})

matches_df = matches_df[matches_df["distance"] <= caliper]

# With replacement

m_treat = df_psm.loc[matches_df["treat_idx"]]
m_control = df_psm.loc[matches_df["control_idx"]]

df_matched = pd.concat([m_treat, m_control])



In [ ]:
treat_spend = m_treat["UTCA"]
control_spend = m_control["UTCA"]

financial_lift = treat_spend.mean() - control_spend.mean()

t, p_value = stats.ttest_ind(treat_spend, control_spend)
display(p_value)
# p = 0.0
display(financial_lift)
# 11854

In [ ]:
%%time

df_p = df_psm[df_psm["app_user"] == 1]
df_n = df_psm[df_psm["app_user"] == 0]

# Sample 300000 for ~15% of data

df_n_common = df_n[df_n["propensity_score"] <= 0.26]
df_n_rare = df_n[df_n["propensity_score"] > 0.26]

df_n_common_downsampled = df_n_common.sample(n = 300000, random_state = 4)
df_n_downsampled = pd.concat([df_n_common_downsampled, df_n_rare])

df_lean = pd.concat([df_p, df_n_downsampled]).reset_index(drop=True)

caliper = 0.2 * df_psm["propensity_score"].std()

# No replacement this time
matcher = NearestNeighborMatch(
    replace = False,
    ratio = 1,
    random_state = 4,
    caliper = 0.2
)

df_matched = matcher.match(
    data = df_lean,
    treatment_col = "app_user",
    score_cols = ["propensity_score"]
)

m_treat = df_matched[df_matched["app_user"] == 1]
m_control = df_matched[df_matched["app_user"] == 0]

treat_spend = m_treat["UTCA"]
control_spend = m_control["UTCA"]

financial_lift = treat_spend.mean() - control_spend.mean()
t, p_value = stats.ttest_ind(treat_spend, control_spend)


In [ ]:

display(f"n treatment:{len(df_p)}")
display(f"matched n treatment:{len(m_treat)}")
display(f"n control:{len(df_n)}")
display(f"mean treatment spend:{treat_spend.mean():.2f}")
display(f"mean control spend:{control_spend.mean():.2f}")
display(f"financial_lift:{financial_lift:.2f}")
display(f"median fl:  {m_treat["UTCA"].median() - m_control["UTCA"].median():.2f}")
display(f"p_value:{p_value:.5e}")

# Did we drop too many treated units because the caliper couldn't find matches?
retention_rate = len(m_treat) / len(df_p) * 100
display(f"{retention_rate:.1f}%")

# Did the downsampled control group maintain the same score extremes?
display(f"Control range:   {df_n["propensity_score"].min():.4f} : {df_n["propensity_score"].max():.4f}")
display(f"downsampled control range:   {df_n_downsampled["propensity_score"].min():.4f} : {df_n_downsampled["propensity_score"].max():.4f}")


'''
The 82.0% retention rate shows the caliper worked. ~35 000 app users dropped.
The downsampled control pool perfects the control range.

Matching with replacement difference is that some high spending legacy were cloned many times
which inflated the control group's average and created the +11854 UTCA.
Without replacement made a 1-to-1 mapping that removed the clones.
Uplift with one to one match: +27892.58 UTCA

The application is a definitively profitable channel that objectively drives higher individual spending compared
to identical users operating on legacy technologies. Median lift proves that the change is data wide and of AT LEAST 8650 ar.
'''


In [ ]:
# Checking distribution of sampled data is accurate

percentiles = [10, 25, 50, 75, 90, 95, 99]

dist_table = pd.DataFrame({
    "control": np.percentile(df_n["propensity_score"], percentiles),
    "downsampled control": np.percentile(df_n_downsampled["propensity_score"], percentiles),
    "treatment": np.percentile(df_p["propensity_score"], percentiles)
}, index = percentiles)

display(dist_table.round(4))

# how many available matches exist at the high end
high_tier_treatment = len(df_p[df_p["propensity_score"] > 0.26])
high_tier_control = len(df_n[df_n["propensity_score"] > 0.26])
high_tier_control_down = len(df_n_downsampled[df_n_downsampled["propensity_score"] > 0.26])

print(f"treatment:{high_tier_treatment}")
print(f"control:{high_tier_control}")
print(f"downsampled control:{high_tier_control_down}")

In [ ]:
%%time

x = df_psm.drop(["UTCA", "app_user", "plans_bought/connectivity__ratio", "avg_transaction_ca"], axis = 1)
y = df_psm["UTCA"]
t = df_psm["app_user"]

x_train, x_test, t_train, t_test, y_train, y_test = train_test_split(x, t, y, test_size = 0.2, random_state = 4)

cols_to_scale = ["connectivity_stability", "n_transactions", "nunique_mobile_plan", "nunique_plan_type",
                "data/voice__ratio"]
cols_to_encod = ["prefered_channel", "prefered_geographic_classification", "prefered_region", "prefered_data_tech"]


scaler_std = StandardScaler()
x_train[cols_to_scale] = scaler_std.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler_std.transform(x_test[cols_to_scale])

encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore").set_output(transform = "pandas")

enc_train = encoder.fit_transform(x_train[cols_to_encod])
x_train = x_train.drop(cols_to_encod, axis = 1).join(enc_train)

enc_test = encoder.transform(x_test[cols_to_encod])
x_test = x_test.drop(cols_to_encod, axis = 1).join(enc_test)


m_causal = xgb.XGBRegressor(
    n_estimators = 300,
    max_depth = 12,
    learning_rate = 0.0165,
    tree_method = "hist",
    n_jobs = -1,
    random_state = 4
)


t_learner = BaseTRegressor(learner = m_causal)


t_learner.fit(X = x_train, treatment = t_train, y = y_train)

ite_app = t_learner.predict(x_test)
ite_train = t_learner.predict(x_train)


In [ ]:
%%time

df_train_ite = x_train.copy()
df_train_ite["predicted_uplift_UTCA"] = ite_train
df_train_ite["is_app_user"] = t_train

df_test_ite = x_test.copy()
df_test_ite["predicted_uplift_UTCA"] = ite_app
df_test_ite["is_app_user"] = t_test

df_ite = pd.concat([df_train_ite, df_test_ite])


df_ite_legacy = df_ite[df_ite["is_app_user"] == 0]

df_ite_legacy = df_ite_legacy.sort_values(by = "predicted_uplift_UTCA", ascending = False)

display(df_ite_legacy.head(10))
display(df_ite_legacy["predicted_uplift_UTCA"].describe())

df_eval = pd.DataFrame({
    "y": y_test,
    "treatment": t_test,
    "model": ite_app.flatten()
})

# Qini curve
plot_qini(df_eval, outcome_col = "y", treatment_col = "treatment", normalize = True)
plt.title("Sum of incremental UTCA uplift")
plt.show()

In [ ]:
'''
The apps do additional revenue for the business but only for a subset of legacy users
Ranking users by their predicted financial lift
so we can find the maximum possible revenue while preventing the customer churn

The model assumes the apps cause the increased spending.

However, the data lacks variables like age, profession, and income.
It is probable that being young and wealthy causes both app adoption and higher spending.
If the model is just capturing "wealth" rather than the App's true causal effect, the predicted lifts will be overestimated.

The data lacks variables information about the phone: a recent smartphone, an older model, or a mobile payment terminal.
Could predict an app uplift for a user who cannot install the apps because they are using an old phone.

The data lacks variables information about data/voice consumption.
The model assumes an increase in purchasing but we don't know if the users are usings what they bought.

The data lacks activation dates so the model treats all customers the same
But old and recent users can help us understand how stable their are

'''

# The graphs tells us that 13% of top uplift legacy users bring addiotnal revenue
# df_ite_legacy.index[: 350000]
#0.13*2679585

with open("legacy_uplifted_msisdn_list.txt", "w") as f:
    for item in df_ite_legacy.index[: 350000]:
        f.write(f"{item}\n")



with open("legacy_uplifted_values_list.txt", "w") as f:
    for item in df_ite_legacy["predicted_uplift_UTCA"][: 350000].to_list():
        f.write(f"{item}\n")

# Understand those user. # k prototypes Clustering

In [ ]:
i = []
with open("legacy_uplifted_msisdn_list.txt", "r") as f:
    line = f.readline().strip()
    while line:
        i.append(line)
        line = f.readline().strip()

top_users_idx = i
df_clustering = df_psm.loc[top_users_idx]


df_clustering = df_clustering.drop(["UTCA", "app_user", "VIP", "propensity_score"], axis = 1) # drop "propensity_score" depending on it is ran


# Untouched: is_multi_plan_user	buys_data	buys_voice	buys_sms
cols_to_scale = ["connectivity_stability", "n_transactions", "nunique_mobile_plan", "nunique_plan_type",
                "data/voice__ratio", "plans_bought/connectivity__ratio", "avg_transaction_ca"]
cols_to_encod = ["prefered_channel", "prefered_geographic_classification", "prefered_region", "prefered_data_tech"]


scaler_std = StandardScaler()
df_clustering[cols_to_scale] = scaler_std.fit_transform(df_clustering[cols_to_scale])


df_clustering_sampled = df_clustering.sample(n = 100000, random_state = 4)

cat_indexes = [df_clustering.columns.get_loc(c) for c in cols_to_encod]





In [ ]:
%%time

costs = []
k_range = range(1, 20)
for k in k_range:
    kproto = KPrototypes(n_clusters = k, n_jobs = -1, random_state = 4)
    kproto.fit(df_clustering_sampled, categorical = cat_indexes)
    costs.append(kproto.cost_)

kl = KneeLocator(
    x = list(k_range),
    y = costs,
    curve = "convex",
    direction = "decreasing"
)

optimal_k = kl.elbow
display(optimal_k)

plt.plot(k_range, costs, marker = "o")
plt.title("Elbow plot for kproto")
plt.xlabel("K clusters")
plt.ylabel("Function cost")
plt.show()

In [ ]:
%%time

# Checking if k = 6 makes sense on sampled data

optimal_k = 6
kproto = KPrototypes(n_clusters = optimal_k, n_jobs = -1, random_state = 4)
labels = kproto.fit_predict(df_clustering_sampled, categorical = cat_indexes)

# Attach clusters
df_eval = df_clustering_sampled.copy()
df_eval["cluster"] = labels

# Size
display(df_eval["cluster"].value_counts(normalize = True))

display(df_eval.groupby("cluster").mean(numeric_only = True))
display(df_eval.groupby("cluster").median(numeric_only = True))

In [ ]:
%%time

# Full data

optimal_k = 6
kproto = KPrototypes(n_clusters = optimal_k, n_jobs = -1, random_state = 4)
labels = kproto.fit_predict(df_clustering, categorical = cat_indexes)

# Attach clusters
df_f_eval = df_clustering.copy()
df_f_eval["cluster"] = labels

In [ ]:
v = []
with open("legacy_uplifted_values_list.txt", "r") as f:
    line = f.readline().strip()
    while line:
        v.append(float(line))
        line = f.readline().strip()

df_f_eval["predicted_uplift_UTCA"] = v


df_f_eval.groupby("cluster").mean(numeric_only = True)

In [ ]:
df_f_eval.groupby("cluster").describe(include = "object")

In [ ]:
with open("cible_ideal_msisdn.txt", "w") as f:
    for item in df_f_eval[df_f_eval["cluster"] == 2].reset_index()["msisdn"].to_list():
        f.write(f"{item}\n")